# UPSC Interview Bias Analysis (2020–2025)

**Research Question:** Do UPSC interview panels systematically award lower marks to candidates from reserved categories (SC, ST, OBC, EWS) compared to General category candidates?

**Data:** Official UPSC Final Marks PDFs, 2020–2025, ~4,500+ candidates.

**Methods:** 10 statistical tests including t-tests, ANOVA, Kruskal-Wallis, Mann-Whitney U, regression, and chi-square; 10 publication-quality visualisations; HTML report.


In [1]:
# ── Cell 0: Install dependencies ────────────────────────────────────────────
import subprocess, sys

pkgs = [
    "pdfplumber", "pandas", "numpy", "scipy",
    "matplotlib", "seaborn", "statsmodels", "openpyxl", "jinja2"
]
print("Installing packages…")
result = subprocess.run(
    [sys.executable, "-m", "pip", "install"] + pkgs,
    capture_output=True, text=True
)
if result.returncode != 0:
    print(result.stderr[-2000:])
else:
    print("All packages installed successfully.")

Installing packages…
All packages installed successfully.


In [6]:
# ── Cell 1: Imports & global config ─────────────────────────────────────────
import os, re, warnings, base64, textwrap
from pathlib import Path
from io import BytesIO

import pdfplumber
import numpy as np
import pandas as pd
import scipy.stats as stats
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import statsmodels.formula.api as smf
from statsmodels.stats.multicomp import pairwise_tukeyhsd
from jinja2 import Template

warnings.filterwarnings("ignore")
matplotlib.rcParams.update({
    "figure.dpi": 150,
    "savefig.dpi": 300,
    "font.family": "DejaVu Sans",
    "axes.spines.top": False,
    "axes.spines.right": False,
})
sns.set_style("whitegrid")

# ── Paths ────────────────────────────────────────────────────────────────────
PROJECT_DIR = Path("/workspaces/upsc-marks-analysis")
OUT_DIR     = PROJECT_DIR / "outputs"
FIG_DIR     = OUT_DIR / "figures"
OUT_DIR.mkdir(exist_ok=True)
FIG_DIR.mkdir(exist_ok=True)

# Canonical categories (order matters for plots)
CATEGORY_ORDER = ["General", "OBC", "SC", "ST", "EWS"]
CATEGORY_COLORS = {
    "General": "#2196F3",
    "OBC":     "#FF9800",
    "SC":      "#F44336",
    "ST":      "#4CAF50",
    "EWS":     "#9C27B0",
}

INTERVIEW_MAX = 275   # UPSC maximum interview marks

print("Imports OK. Output dirs:", OUT_DIR)

Imports OK. Output dirs: /workspaces/upsc-marks-analysis/outputs


In [7]:
# ── Cell 2: PDF Extraction ───────────────────────────────────────────────────
#
# UPSC Final Marks PDF format (all years):
#   - Landscape, word-based layout (no auto-detected tables)
#   - Data rows start with a 7-digit ROLL_NO
#   - Last token = S.NO (rank, sequential integer)
#   - 3 mark columns: W_TOTAL, PT_MARKS (interview), F_TOTAL
#   - COMM (category) appears BEFORE marks in 2021–2025
#                            AFTER  marks in 2020
#   - PwBD-X tag may appear alongside category or alone
# ── helpers ──────────────────────────────────────────────────────────────────
import re

ROLL_RE     = re.compile(r"^\d{6,8}$")
NUMERIC_RE  = re.compile(r"^\d+\.?\d*$")
# Matches recognised category/PwBD codes
CAT_RE = re.compile(
    r"^(GEN|GENERAL|UR|OBC(?:-NCL|-A)?|SC|ST|EWS|PwBD-\d+)$",
    re.IGNORECASE,
)

CATEGORY_ALIASES = {
    "GEN": "General", "GENERAL": "General", "UR": "General",
    "OBC": "OBC", "OBC-NCL": "OBC", "OBC-A": "OBC",
    "SC": "SC", "ST": "ST", "EWS": "EWS",
}


def normalise_category(raw: str) -> str:
    if not raw:
        return "General"
    key = raw.strip().upper()
    # Strip PwBD suffixes
    if key.startswith("PWBD"):
        return "PH"
    return CATEGORY_ALIASES.get(key, CATEGORY_ALIASES.get(key.replace("-", ""), key))


def parse_data_row(tokens: list) -> dict | None:
    """
    Parse a single data row from a UPSC marks PDF.

    Format (all years):
        ROLL_NO  NAME_PARTS...  [COMM]  [PwBD-X]  W_TOTAL  PT_MARKS  F_TOTAL  S.NO
    or (2020 for reserved):
        ROLL_NO  NAME_PARTS...  W_TOTAL  PT_MARKS  F_TOTAL  [COMM]  [PwBD-X]  S.NO
    """
    if len(tokens) < 6:
        return None
    if not ROLL_RE.match(tokens[0]):
        return None

    roll_no   = tokens[0]
    remaining = tokens[1:]
    n = len(remaining)

    # ── Step 1: strip S.NO from the right ────────────────────────────────────
    try:
        sno = int(remaining[-1])   # must be a clean integer
    except ValueError:
        return None
    remaining = remaining[:-1]

    # ── Step 2: collect trailing non-numeric tokens (could be COMM/PwBD in 2020) ──
    trailing_cats = []
    while remaining and CAT_RE.match(remaining[-1]):
        trailing_cats.insert(0, remaining[-1])
        remaining = remaining[:-1]

    # ── Step 3: collect 3 numeric marks from right ───────────────────────────
    marks = []
    while remaining and NUMERIC_RE.match(remaining[-1]) and len(marks) < 3:
        marks.insert(0, float(remaining[-1]))
        remaining = remaining[:-1]

    if len(marks) != 3:
        return None
    w_total, pt_marks, f_total = marks

    # Basic sanity check
    if not (0 <= pt_marks <= 275):
        return None

    # ── Step 4: collect leading category tokens (COMM/PwBD in 2021+) ─────────
    leading_cats = []
    while remaining and CAT_RE.match(remaining[-1]):
        leading_cats.insert(0, remaining[-1])
        remaining = remaining[:-1]

    name = " ".join(remaining).strip()

    # ── Step 5: resolve category ──────────────────────────────────────────────
    all_cats = leading_cats + trailing_cats
    cat_codes = [c for c in all_cats if not c.upper().startswith("PWBD")]
    is_pwbd   = any(c.upper().startswith("PWBD") for c in all_cats)

    raw_cat  = cat_codes[0] if cat_codes else "GEN"
    category = normalise_category(raw_cat)

    if not name or not roll_no:
        return None

    return {
        "roll_no":      roll_no,
        "name":         name,
        "category":     category,
        "written_total": w_total,
        "interview":    pt_marks,
        "final_total":  f_total,
        "is_pwbd":      is_pwbd,
    }


def extract_rows_from_page(page) -> list[list[str]]:
    """
    Extract word-grouped rows from a pdfplumber page.
    Groups words by approximate y-position.
    """
    words = page.extract_words(x_tolerance=5, y_tolerance=5)
    if not words:
        return []
    rows_dict: dict[int, list] = {}
    for w in words:
        y_key = round(w["top"] / 5) * 5
        rows_dict.setdefault(y_key, []).append(w)
    rows = []
    for y_key in sorted(rows_dict):
        row_words = sorted(rows_dict[y_key], key=lambda w: w["x0"])
        rows.append([w["text"] for w in row_words])
    return rows


def parse_pdf(pdf_path: Path, year: int) -> pd.DataFrame:
    """
    Extract candidate data from a single UPSC marks PDF.
    Returns a DataFrame with columns:
        roll_no, name, category, written_total, interview, final_total, year
    """
    records = []
    print(f"  {pdf_path.name} … ", end="", flush=True)

    with pdfplumber.open(pdf_path) as pdf:
        total_pages = len(pdf.pages)
        for page in pdf.pages:
            rows = extract_rows_from_page(page)
            for tokens in rows:
                rec = parse_data_row(tokens)
                if rec:
                    rec["year"] = year
                    records.append(rec)

    df = pd.DataFrame(records)
    print(f"{len(df)} records from {total_pages} pages.")
    return df


# ── Main extraction loop ──────────────────────────────────────────────────────
PDF_YEAR_MAP = {
    "upsc_marks_2020.pdf":      2020,
    "upsc_marks_2021.pdf":      2021,
    "upsc_marks_2022.pdf":      2022,
    "upsc_marks_2023.pdf":      2023,
    "upsc_marks_2024.pdf":      2024,
    "UPSC-Marks-2025_copy.pdf": 2025,
}

all_frames = []
print("=" * 60)
print("UPSC PDF Extraction")
print("=" * 60)

for filename, year in PDF_YEAR_MAP.items():
    pdf_path = PROJECT_DIR / filename
    if not pdf_path.exists():
        print(f"  [SKIP] {filename} not found.")
        continue
    try:
        df_year = parse_pdf(pdf_path, year)
        if not df_year.empty:
            all_frames.append(df_year)
    except Exception as e:
        print(f"  [ERROR] {filename}: {e}")

if not all_frames:
    raise RuntimeError("No data extracted. Check PDF paths.")

raw_df = pd.concat(all_frames, ignore_index=True)
raw_df.to_csv(OUT_DIR / "raw_data.csv", index=False)
print(f"\nRaw data → outputs/raw_data.csv  ({len(raw_df):,} records)")
print(raw_df.groupby("year")["roll_no"].count().rename("n_records"))
print("\nCategory breakdown (raw):")
print(raw_df.groupby(["year", "category"])["roll_no"].count().unstack(fill_value=0))


UPSC PDF Extraction
  upsc_marks_2020.pdf … 761 records from 39 pages.
  upsc_marks_2021.pdf … 685 records from 39 pages.
  upsc_marks_2022.pdf … 933 records from 55 pages.
  upsc_marks_2023.pdf … 1016 records from 55 pages.
  upsc_marks_2024.pdf … 999 records from 57 pages.
  UPSC-Marks-2025_copy.pdf … 958 records from 38 pages.

Raw data → outputs/raw_data.csv  (5,352 records)
year
2020     761
2021     685
2022     933
2023    1016
2024     999
2025     958
Name: n_records, dtype: int64

Category breakdown (raw):
category  EWS  General  OBC   SC  ST
year                                
2020       86      263  229  122  61
2021       73      244  203  105  60
2022       99      345  263  154  72
2023      115      347  303  165  86
2024      109      328  315  160  87
2025      104      317  306  158  73


In [8]:
# ── MANUAL OVERRIDE (optional) ───────────────────────────────────────────────
# If auto-detection failed for a year, you can manually specify column indices.
# Uncomment and set the correct indices; re-run this cell and the next.
#
# MANUAL_COLUMN_OVERRIDES = {
#     2022: {"roll": 0, "name": 1, "category": 2, "written": 14, "interview": 15, "total": 16},
# }
# Then re-run the extraction cell with these overrides applied.

print("Manual override cell — no action taken. Edit above if needed.")

Manual override cell — no action taken. Edit above if needed.


In [9]:
# ── Cell 3: Data Validation & Cleaning ──────────────────────────────────────

report_lines = ["UPSC Interview Bias Study — Data Quality Report",
                "=" * 55, ""]

df = raw_df.copy()
report_lines.append(f"Total raw records: {len(df):,}")
report_lines.append("")

# ── Per-year counts ───────────────────────────────────────────────────────────
report_lines.append("Records per year (raw):")
for yr, cnt in df["year"].value_counts().sort_index().items():
    report_lines.append(f"  {yr}: {cnt:>5,}")
report_lines.append("")

# ── Remove duplicates ─────────────────────────────────────────────────────────
before = len(df)
df = df.drop_duplicates(subset=["roll_no", "year"])
report_lines.append(f"Duplicate rows removed: {before - len(df):,}")

# ── Drop rows with missing interview or category ──────────────────────────────
missing_int  = df["interview"].isna().sum()
missing_cat  = df["category"].isna().sum()
report_lines.append(f"Missing interview marks: {missing_int:,}")
report_lines.append(f"Missing category:        {missing_cat:,}")

df = df.dropna(subset=["interview", "category"])
report_lines.append(f"Records after dropping missing: {len(df):,}")
report_lines.append("")

# ── Validate interview range 0–275 ────────────────────────────────────────────
out_of_range = df[(df["interview"] < 0) | (df["interview"] > INTERVIEW_MAX)]
report_lines.append(f"Interview out of range [0–{INTERVIEW_MAX}]: {len(out_of_range):,}")
df = df[(df["interview"] >= 0) & (df["interview"] <= INTERVIEW_MAX)]

# ── Keep only known main categories ──────────────────────────────────────────
valid_cats = {"General", "OBC", "SC", "ST", "EWS"}
excluded_cats = df[~df["category"].isin(valid_cats)]["category"].value_counts()
if not excluded_cats.empty:
    report_lines.append("Excluded categories (e.g., PH/unknown):")
    for cat, n in excluded_cats.items():
        report_lines.append(f"  {cat}: {n}")
df = df[df["category"].isin(valid_cats)].copy()

# ── Ensure category is ordered categorical ────────────────────────────────────
df["category"] = pd.Categorical(df["category"], categories=CATEGORY_ORDER, ordered=False)

report_lines.append("")
report_lines.append(f"Clean records (final): {len(df):,}")
report_lines.append("")

# ── Per-year × per-category counts ───────────────────────────────────────────
report_lines.append("Records per year × category (clean):")
cross = df.groupby(["year", "category"], observed=True).size().unstack(fill_value=0)
report_lines.append(cross.to_string())
report_lines.append("")

# ── Missing rate summary ──────────────────────────────────────────────────────
report_lines.append("Missing rate per column (clean dataset):")
for col in ["written_total", "interview", "final_total", "category"]:
    pct = 100 * df[col].isna().sum() / len(df)
    report_lines.append(f"  {col}: {pct:.1f}%")

# ── Save ──────────────────────────────────────────────────────────────────────
df.to_csv(OUT_DIR / "cleaned_data.csv", index=False)
report_text = "\n".join(report_lines)
(OUT_DIR / "data_quality_report.txt").write_text(report_text)

print(report_text)

UPSC Interview Bias Study — Data Quality Report

Total raw records: 5,352

Records per year (raw):
  2020:   761
  2021:   685
  2022:   933
  2023: 1,016
  2024:   999
  2025:   958

Duplicate rows removed: 0
Missing interview marks: 0
Missing category:        0
Records after dropping missing: 5,352

Interview out of range [0–275]: 0

Clean records (final): 5,352

Records per year × category (clean):
category  General  OBC   SC  ST  EWS
year                                
2020          263  229  122  61   86
2021          244  203  105  60   73
2022          345  263  154  72   99
2023          347  303  165  86  115
2024          328  315  160  87  109
2025          317  306  158  73  104

Missing rate per column (clean dataset):
  written_total: 0.0%
  interview: 0.0%
  final_total: 0.0%
  category: 0.0%


In [10]:
# ── Cell 4: Descriptive Statistics ───────────────────────────────────────────

def desc_stats_group(g):
    """Compute descriptive statistics for a group of interview marks."""
    x = g.dropna().values
    n = len(x)
    if n < 2:
        return pd.Series({k: np.nan for k in
                          ["n", "mean", "median", "std", "min", "max",
                           "q1", "q3", "iqr", "skew", "kurt", "ci_lo", "ci_hi"]})
    m = np.mean(x)
    se = stats.sem(x)
    ci = stats.t.interval(0.95, df=n - 1, loc=m, scale=se)
    q1, q3 = np.percentile(x, [25, 75])
    return pd.Series({
        "n":      n,
        "mean":   round(m, 2),
        "median": round(np.median(x), 2),
        "std":    round(np.std(x, ddof=1), 2),
        "min":    np.min(x),
        "max":    np.max(x),
        "q1":     q1,
        "q3":     q3,
        "iqr":    q3 - q1,
        "skew":   round(stats.skew(x), 3),
        "kurt":   round(stats.kurtosis(x), 3),
        "ci_lo":  round(ci[0], 2),
        "ci_hi":  round(ci[1], 2),
    })


desc_rows = []

# Per year × category
for (yr, cat), g in df.groupby(["year", "category"], observed=True)["interview"]:
    row = desc_stats_group(g)
    row["year"] = yr
    row["category"] = cat
    row["scope"] = "year"
    desc_rows.append(row)

# Pooled across years per category
for cat, g in df.groupby("category", observed=True)["interview"]:
    row = desc_stats_group(g)
    row["year"] = "pooled"
    row["category"] = cat
    row["scope"] = "pooled"
    desc_rows.append(row)

desc_df = pd.DataFrame(desc_rows)[["scope", "year", "category", "n",
                                    "mean", "median", "std",
                                    "q1", "q3", "iqr",
                                    "skew", "kurt",
                                    "ci_lo", "ci_hi",
                                    "min", "max"]]
desc_df.to_csv(OUT_DIR / "descriptive_stats.csv", index=False)

# ── Interview mark gap ────────────────────────────────────────────────────────
gap_rows = []
for yr in df["year"].unique():
    gen_mean = df[(df["year"] == yr) & (df["category"] == "General")]["interview"].mean()
    for cat in ["OBC", "SC", "ST", "EWS"]:
        res_mean = df[(df["year"] == yr) & (df["category"] == cat)]["interview"].mean()
        gap_rows.append({"year": yr, "category": cat,
                         "general_mean": round(gen_mean, 2),
                         "category_mean": round(res_mean, 2),
                         "gap_general_minus_cat": round(gen_mean - res_mean, 2)})
gap_df = pd.DataFrame(gap_rows)

# ── Spearman correlation: written rank vs interview rank ──────────────────────
spearman_rows = []
for yr in sorted(df["year"].unique()):
    sub = df[(df["year"] == yr)].dropna(subset=["written_total", "interview"])
    if len(sub) > 5:
        rho, pval = stats.spearmanr(sub["written_total"], sub["interview"])
        spearman_rows.append({"year": yr, "spearman_rho": round(rho, 3), "p_value": round(pval, 4)})
spearman_df = pd.DataFrame(spearman_rows)

print("Descriptive stats saved.")
print("\nMean interview marks by category (pooled):")
pooled = desc_df[desc_df["scope"] == "pooled"][["category", "n", "mean", "std", "ci_lo", "ci_hi"]]
print(pooled.to_string(index=False))

print("\nInterview mark gap (General − reserved) by year:")
print(gap_df.to_string(index=False))

print("\nSpearman ρ: written vs interview marks")
print(spearman_df.to_string(index=False))

Descriptive stats saved.

Mean interview marks by category (pooled):
category      n   mean   std  ci_lo  ci_hi
 General 1844.0 181.16 16.73 180.39 181.92
     OBC 1619.0 175.42 16.41 174.62 176.22
      SC  864.0 172.32 17.62 171.15 173.50
      ST  439.0 171.90 18.03 170.21 173.59
     EWS  586.0 174.50 16.18 173.19 175.82

Interview mark gap (General − reserved) by year:
 year category  general_mean  category_mean  gap_general_minus_cat
 2020      OBC        178.23         173.03                   5.21
 2020       SC        178.23         166.17                  12.06
 2020       ST        178.23         165.85                  12.38
 2020      EWS        178.23         169.10                   9.13
 2021      OBC        174.66         169.36                   5.30
 2021       SC        174.66         166.88                   7.79
 2021       ST        174.66         161.23                  13.43
 2021      EWS        174.66         166.82                   7.84
 2022      OBC      

In [11]:
# ── Cell 5: Statistical Tests ─────────────────────────────────────────────────

def cohens_d(x, y):
    """Cohen's d for two independent samples."""
    nx, ny = len(x), len(y)
    if nx < 2 or ny < 2:
        return np.nan
    pooled_std = np.sqrt(((nx - 1) * np.var(x, ddof=1) + (ny - 1) * np.var(y, ddof=1)) / (nx + ny - 2))
    return (np.mean(x) - np.mean(y)) / pooled_std if pooled_std > 0 else 0.0


def eta_squared(f_stat, df_between, df_within):
    """η² from F-statistic."""
    ss_between = f_stat * df_between
    ss_total = ss_between + df_within
    return ss_between / ss_total if ss_total > 0 else np.nan


def rank_biserial_r(u_stat, n1, n2):
    """Rank-biserial correlation (effect size for Mann-Whitney U)."""
    return 1 - (2 * u_stat) / (n1 * n2) if n1 * n2 > 0 else np.nan


test_results = []

# ── Helper to run tests on a subset ──────────────────────────────────────────

def run_all_tests(sub_df, scope_label):
    """Run all 10 statistical tests on a (sub)dataframe."""

    groups = {cat: sub_df[sub_df["category"] == cat]["interview"].dropna().values
              for cat in CATEGORY_ORDER}
    available = {cat: g for cat, g in groups.items() if len(g) >= 5}
    gen = available.get("General", np.array([]))
    reserved_combined = np.concatenate([g for cat, g in available.items() if cat != "General"])
    all_arrays = list(available.values())

    # 1. Levene's test
    if len(all_arrays) >= 2:
        stat, p = stats.levene(*all_arrays)
        test_results.append({
            "scope": scope_label, "test": "Levene's Test",
            "statistic": round(stat, 4), "p_value": round(p, 4),
            "effect_size": np.nan, "effect_size_type": "",
            "interpretation": ("Variances differ significantly across categories (p<0.05)."
                               if p < 0.05 else
                               "No significant difference in variances (p≥0.05).")
        })

    # 2. Shapiro-Wilk per category
    for cat, g in available.items():
        if len(g) < 3:
            continue
        g_sample = g[:5000]  # Shapiro-Wilk max n ≈ 5000
        stat, p = stats.shapiro(g_sample)
        test_results.append({
            "scope": scope_label, "test": f"Shapiro-Wilk [{cat}]",
            "statistic": round(stat, 4), "p_value": round(p, 4),
            "effect_size": np.nan, "effect_size_type": "",
            "interpretation": (f"{cat} interview marks are NOT normally distributed (p<0.05)."
                               if p < 0.05 else
                               f"{cat} interview marks are approximately normal (p≥0.05).")
        })

    # 3. Independent t-test: General vs all reserved combined
    if len(gen) >= 5 and len(reserved_combined) >= 5:
        stat, p = stats.ttest_ind(gen, reserved_combined, equal_var=False)
        d = cohens_d(gen, reserved_combined)
        test_results.append({
            "scope": scope_label, "test": "Welch t-test (General vs Reserved)",
            "statistic": round(stat, 4), "p_value": round(p, 4),
            "effect_size": round(d, 3), "effect_size_type": "Cohen's d",
            "interpretation": (
                f"General mean ({np.mean(gen):.1f}) is {'significantly' if p<0.05 else 'NOT significantly'} "
                f"different from reserved mean ({np.mean(reserved_combined):.1f}) — "
                f"d={d:.2f} ({'large' if abs(d)>=0.8 else 'medium' if abs(d)>=0.5 else 'small'} effect)."
            )
        })

    # 4. One-way ANOVA
    if len(all_arrays) >= 2:
        stat, p = stats.f_oneway(*all_arrays)
        n_groups = len(all_arrays)
        n_total  = sum(len(g) for g in all_arrays)
        es = eta_squared(stat, n_groups - 1, n_total - n_groups)
        test_results.append({
            "scope": scope_label, "test": "One-way ANOVA",
            "statistic": round(stat, 4), "p_value": round(p, 4),
            "effect_size": round(es, 4), "effect_size_type": "η²",
            "interpretation": (
                f"{'Significant' if p<0.05 else 'No significant'} difference in mean interview marks "
                f"across categories (F={stat:.2f}, p={p:.4f}, η²={es:.3f})."
            )
        })

    # 5. Tukey HSD
    if len(available) >= 2:
        try:
            data_vals = np.concatenate(list(available.values()))
            data_labs = np.concatenate([[cat] * len(g) for cat, g in available.items()])
            tukey = pairwise_tukeyhsd(data_vals, data_labs, alpha=0.05)
            sig_pairs = [(str(r[0]), str(r[1])) for r in tukey.summary().data[1:] if r[5] <= 0.05]
            test_results.append({
                "scope": scope_label, "test": "Tukey HSD (post-hoc)",
                "statistic": np.nan, "p_value": np.nan,
                "effect_size": np.nan, "effect_size_type": "",
                "interpretation": (
                    f"Significant pairs (p≤0.05): {sig_pairs}" if sig_pairs else
                    "No significant pairwise differences after Tukey correction."
                )
            })
        except Exception as e:
            test_results.append({
                "scope": scope_label, "test": "Tukey HSD (post-hoc)",
                "statistic": np.nan, "p_value": np.nan,
                "effect_size": np.nan, "effect_size_type": "",
                "interpretation": f"Tukey HSD could not be computed: {e}"
            })

    # 6. Kruskal-Wallis
    if len(all_arrays) >= 2:
        stat, p = stats.kruskal(*all_arrays)
        test_results.append({
            "scope": scope_label, "test": "Kruskal-Wallis",
            "statistic": round(stat, 4), "p_value": round(p, 4),
            "effect_size": np.nan, "effect_size_type": "",
            "interpretation": (
                f"{'Significant' if p<0.05 else 'No significant'} distributional difference "
                f"across categories (H={stat:.2f}, p={p:.4f})."
            )
        })

    # 7. Mann-Whitney U: General vs each reserved
    for cat in ["OBC", "SC", "ST", "EWS"]:
        res = available.get(cat, np.array([]))
        if len(gen) < 5 or len(res) < 5:
            continue
        stat, p = stats.mannwhitneyu(gen, res, alternative="two-sided")
        r = rank_biserial_r(stat, len(gen), len(res))
        test_results.append({
            "scope": scope_label, "test": f"Mann-Whitney U (General vs {cat})",
            "statistic": round(stat, 4), "p_value": round(p, 4),
            "effect_size": round(r, 3), "effect_size_type": "rank-biserial r",
            "interpretation": (
                f"General {'significantly higher' if p<0.05 and np.mean(gen)>np.mean(res) else 'NOT significantly different'} "
                f"than {cat} (U={stat:.0f}, p={p:.4f}, r={r:.2f})."
            )
        })

    # 8. Chi-square: selection proportions
    # Compare observed category distribution vs proportional expectation
    cat_counts = sub_df["category"].value_counts()
    if len(cat_counts) >= 2:
        observed = cat_counts.values
        expected = np.full(len(observed), observed.mean())  # null: equal representation
        stat, p = stats.chisquare(observed, f_exp=expected)
        test_results.append({
            "scope": scope_label, "test": "Chi-square (category representation)",
            "statistic": round(stat, 4), "p_value": round(p, 4),
            "effect_size": np.nan, "effect_size_type": "",
            "interpretation": (
                f"Category selection distribution {'significantly' if p<0.05 else 'NOT significantly'} "
                f"deviates from equal representation (χ²={stat:.2f}, p={p:.4f})."
            )
        })

    # 9. Simple regression: written → interview, by category
    for cat, g_cat in available.items():
        sub_cat = sub_df[(sub_df["category"] == cat)].dropna(subset=["written_total", "interview"])
        if len(sub_cat) < 5:
            continue
        slope, intercept, r_val, p_val, se = stats.linregress(
            sub_cat["written_total"], sub_cat["interview"]
        )
        test_results.append({
            "scope": scope_label, "test": f"Linear regression written→interview [{cat}]",
            "statistic": round(slope, 4), "p_value": round(p_val, 4),
            "effect_size": round(r_val ** 2, 3), "effect_size_type": "R²",
            "interpretation": (
                f"For {cat}: slope={slope:.3f} (p={p_val:.4f}), R²={r_val**2:.3f}. "
                f"{'Significant' if p_val<0.05 else 'Non-significant'} written→interview relationship."
            )
        })

    # 10. Multiple regression: interview ~ written + category + year
    reg_df = sub_df.dropna(subset=["written_total", "interview"]).copy()
    reg_df["category_str"] = reg_df["category"].astype(str)
    if len(reg_df) >= 20 and reg_df["category_str"].nunique() >= 2:
        try:
            formula = "interview ~ written_total + C(category_str) + C(year)"
            model = smf.ols(formula, data=reg_df).fit()
            test_results.append({
                "scope": scope_label, "test": "OLS Multiple Regression",
                "statistic": round(model.fvalue, 4), "p_value": round(model.f_pvalue, 4),
                "effect_size": round(model.rsquared_adj, 3), "effect_size_type": "Adj. R²",
                "interpretation": (
                    f"Model F={model.fvalue:.2f} (p={model.f_pvalue:.4f}), Adj.R²={model.rsquared_adj:.3f}. "
                    f"Category coefficients (vs General): "
                    + ", ".join(
                        f"{k.replace('C(category_str)[T.', '').rstrip(']')}={v:.2f}"
                        for k, v in model.params.items() if "category_str" in k
                    )
                )
            })
        except Exception as e:
            test_results.append({
                "scope": scope_label, "test": "OLS Multiple Regression",
                "statistic": np.nan, "p_value": np.nan,
                "effect_size": np.nan, "effect_size_type": "",
                "interpretation": f"Regression failed: {e}"
            })


# ── Run tests on pooled data ──────────────────────────────────────────────────
print("Running tests on POOLED dataset…")
run_all_tests(df, "pooled")

# ── Run tests per year ────────────────────────────────────────────────────────
for yr in sorted(df["year"].unique()):
    print(f"Running tests for {yr}…")
    run_all_tests(df[df["year"] == yr], str(yr))

tests_df = pd.DataFrame(test_results)
tests_df.to_csv(OUT_DIR / "statistical_tests.csv", index=False)
print(f"\nStatistical tests complete. {len(tests_df)} results saved.")
print("\nKey pooled results:")
key_tests = ["Welch t-test (General vs Reserved)", "One-way ANOVA", "Kruskal-Wallis", "OLS Multiple Regression"]
print(tests_df[(tests_df["scope"] == "pooled") & (tests_df["test"].isin(key_tests))]
      [["test", "statistic", "p_value", "effect_size", "interpretation"]].to_string(index=False))

Running tests on POOLED dataset…
Running tests for 2020…
Running tests for 2021…
Running tests for 2022…
Running tests for 2023…
Running tests for 2024…
Running tests for 2025…

Statistical tests complete. 147 results saved.

Key pooled results:
                              test  statistic  p_value  effect_size                                                                                                         interpretation
Welch t-test (General vs Reserved)    14.6746      0.0       0.4200                    General mean (181.2) is significantly different from reserved mean (174.1) — d=0.42 (small effect).
                     One-way ANOVA    60.5077      0.0       0.0433                        Significant difference in mean interview marks across categories (F=60.51, p=0.0000, η²=0.043).
                    Kruskal-Wallis   226.5733      0.0          NaN                                          Significant distributional difference across categories (H=226.57, p=0.0000).
      

In [12]:
print("=" * 70)
print("CELL 6: Written-Score Quintile Stratification")
print("=" * 70)

CELL 6: Written-Score Quintile Stratification


In [13]:
# Step 1: Create global quintiles (all candidates ranked together)
df["written_quintile"] = pd.qcut(
    df["written_total"], q=5,
    labels=["Q1 (Bottom 20%)", "Q2 (20-40%)", "Q3 (40-60%)", "Q4 (60-80%)", "Q5 (Top 20%)"]
)
 
Q_LABELS = ["Q1 (Bottom 20%)", "Q2 (20-40%)", "Q3 (40-60%)", "Q4 (60-80%)", "Q5 (Top 20%)"]
Q_SHORT  = ["Q1", "Q2", "Q3", "Q4", "Q5"]
 

In [14]:
# Step 2: Check category distribution per quintile
# (Reserved categories cluster in lower quintiles due to lower cut-offs)
print("\n── Category distribution across quintiles ──")
quintile_dist = df.groupby(["written_quintile", "category"]).size().unstack(fill_value=0)
quintile_dist = quintile_dist[CATEGORY_ORDER]
print(quintile_dist)
 


── Category distribution across quintiles ──
category          General  OBC   SC   ST  EWS
written_quintile                             
Q1 (Bottom 20%)        83  267  428  233   75
Q2 (20-40%)           138  471  222  102  167
Q3 (40-60%)           329  430  109   47  142
Q4 (60-80%)           562  268   69   43  128
Q5 (Top 20%)          732  183   36   14   74


In [15]:
# Step 3: Compute descriptive stats per (quintile × category)
print("\n── Interview marks by quintile × category ──")
quintile_stats = (
    df.groupby(["written_quintile", "category"])["interview"]
    .agg(["count", "mean", "std"])
    .round(2)
)
quintile_stats["ci_lo"] = (quintile_stats["mean"] - 1.96 * quintile_stats["std"] / np.sqrt(quintile_stats["count"])).round(2)
quintile_stats["ci_hi"] = (quintile_stats["mean"] + 1.96 * quintile_stats["std"] / np.sqrt(quintile_stats["count"])).round(2)
print(quintile_stats)
 


── Interview marks by quintile × category ──
                           count    mean    std   ci_lo   ci_hi
written_quintile category                                      
Q1 (Bottom 20%)  General      83  163.22  21.85  158.52  167.92
                 OBC         267  187.49  15.41  185.64  189.34
                 SC          428  177.74  15.25  176.30  179.18
                 ST          233  178.26  15.56  176.26  180.26
                 EWS          75  186.48  15.60  182.95  190.01
Q2 (20-40%)      General     138  196.47  13.91  194.15  198.79
                 OBC         471  178.10  12.92  176.93  179.27
                 SC          222  167.87  17.79  165.53  170.21
                 ST          102  165.11  17.22  161.77  168.45
                 EWS         167  179.51  12.80  177.57  181.45
Q3 (40-60%)      General     329  191.16  10.91  189.98  192.34
                 OBC         430  172.03  14.50  170.66  173.40
                 SC          109  165.84  18.04  162.45  1

In [16]:
# Step 4: Compute gap from General within each quintile
print("\n── Gap from General within each quintile ──")
quintile_means = df.groupby(["written_quintile", "category"])["interview"].mean().unstack()
for cat in ["OBC", "SC", "ST", "EWS"]:
    if cat in quintile_means.columns:
        quintile_means[f"Gap_Gen_vs_{cat}"] = (quintile_means["General"] - quintile_means[cat]).round(2)
print(quintile_means[[c for c in quintile_means.columns if "Gap" in c]])
 


── Gap from General within each quintile ──
category          Gap_Gen_vs_OBC  Gap_Gen_vs_SC  Gap_Gen_vs_ST  Gap_Gen_vs_EWS
written_quintile                                                              
Q1 (Bottom 20%)           -24.28         -14.52         -15.04          -23.26
Q2 (20-40%)                18.38          28.60          31.36           16.96
Q3 (40-60%)                19.13          25.32          27.46           20.61
Q4 (60-80%)                13.43          14.70          13.88           12.78
Q5 (Top 20%)                5.50           9.24          18.06            6.34


In [17]:
# Step 5: Statistical tests within each quintile (Mann-Whitney U, General vs each)
print("\n── Within-quintile Mann-Whitney U tests (General vs each reserved) ──")
quintile_test_results = []
 
for q_label, q_short in zip(Q_LABELS, Q_SHORT):
    q_df = df[df["written_quintile"] == q_label]
    gen_vals = q_df[q_df["category"] == "General"]["interview"].values
 
    for cat in ["OBC", "SC", "ST", "EWS"]:
        cat_vals = q_df[q_df["category"] == cat]["interview"].values
 
        # Skip if either group is too small for a meaningful test
        if len(gen_vals) < 10 or len(cat_vals) < 10:
            quintile_test_results.append({
                "quintile": q_short, "comparison": f"General vs {cat}",
                "n_gen": len(gen_vals), "n_cat": len(cat_vals),
                "gen_mean": round(np.mean(gen_vals), 1) if len(gen_vals) > 0 else np.nan,
                "cat_mean": round(np.mean(cat_vals), 1) if len(cat_vals) > 0 else np.nan,
                "gap": np.nan, "cohens_d": np.nan,
                "U": np.nan, "p_value": np.nan, "rank_biserial_r": np.nan,
                "significant": False, "note": "Insufficient sample size"
            })
            continue
 
        # Mann-Whitney U
        u_stat, p_val = stats.mannwhitneyu(gen_vals, cat_vals, alternative="two-sided")
        r = rank_biserial_r(u_stat, len(gen_vals), len(cat_vals))
        d = cohens_d(gen_vals, cat_vals)
        gap = round(np.mean(gen_vals) - np.mean(cat_vals), 1)
 
        quintile_test_results.append({
            "quintile": q_short, "comparison": f"General vs {cat}",
            "n_gen": len(gen_vals), "n_cat": len(cat_vals),
            "gen_mean": round(np.mean(gen_vals), 1),
            "cat_mean": round(np.mean(cat_vals), 1),
            "gap": gap, "cohens_d": round(d, 3),
            "U": round(u_stat, 1), "p_value": round(p_val, 6),
            "rank_biserial_r": round(r, 3),
            "significant": p_val < 0.05, "note": ""
        })
 
quintile_tests_df = pd.DataFrame(quintile_test_results)
quintile_tests_df.to_csv(OUT_DIR / "quintile_stratification_tests.csv", index=False)
 
# Print key results (Q2-Q5 where both groups have adequate samples)
print("\nKey results (Q2–Q5, where General and reserved groups overlap):")
mask = quintile_tests_df["quintile"].isin(["Q2", "Q3", "Q4", "Q5"]) & quintile_tests_df["significant"]
print(quintile_tests_df[mask][["quintile", "comparison", "gap", "cohens_d", "p_value"]].to_string(index=False))
 


── Within-quintile Mann-Whitney U tests (General vs each reserved) ──

Key results (Q2–Q5, where General and reserved groups overlap):
quintile     comparison  gap  cohens_d  p_value
      Q2 General vs OBC 18.4     1.398 0.000000
      Q2  General vs SC 28.6     1.743 0.000000
      Q2  General vs ST 31.4     2.036 0.000000
      Q2 General vs EWS 17.0     1.274 0.000000
      Q3 General vs OBC 19.1     1.464 0.000000
      Q3  General vs SC 25.3     1.941 0.000000
      Q3  General vs ST 27.5     2.336 0.000000
      Q3 General vs EWS 20.6     1.621 0.000000
      Q4 General vs OBC 13.4     0.902 0.000000
      Q4  General vs SC 14.7     0.964 0.000000
      Q4  General vs ST 13.9     0.930 0.000001
      Q4 General vs EWS 12.8     0.892 0.000000
      Q5 General vs OBC  5.5     0.331 0.000961
      Q5  General vs SC  9.2     0.584 0.000799
      Q5  General vs ST 18.1     1.137 0.000722
      Q5 General vs EWS  6.3     0.397 0.011966


In [18]:
# Step 6: Kruskal-Wallis within each quintile (all 5 categories)
print("\n── Within-quintile Kruskal-Wallis (all categories) ──")
for q_label, q_short in zip(Q_LABELS, Q_SHORT):
    q_df = df[df["written_quintile"] == q_label]
    groups = [q_df[q_df["category"] == cat]["interview"].values
              for cat in CATEGORY_ORDER
              if len(q_df[q_df["category"] == cat]) >= 5]
    if len(groups) >= 2:
        h_stat, p_val = stats.kruskal(*groups)
        sig = "***" if p_val < 0.001 else "**" if p_val < 0.01 else "*" if p_val < 0.05 else "ns"
        print(f"  {q_short}: H={h_stat:.2f}, p={p_val:.6f} {sig}")
 
print(f"\n✓ Quintile analysis complete. {len(quintile_tests_df)} tests saved to quintile_stratification_tests.csv")
 


── Within-quintile Kruskal-Wallis (all categories) ──
  Q1: H=147.11, p=0.000000 ***
  Q2: H=275.81, p=0.000000 ***
  Q3: H=360.83, p=0.000000 ***
  Q4: H=171.75, p=0.000000 ***
  Q5: H=32.84, p=0.000001 ***

✓ Quintile analysis complete. 20 tests saved to quintile_stratification_tests.csv


In [19]:
print("\n" + "=" * 70)
print("CELL 7: Gap Stability Test (Category × Year Interaction)")
print("=" * 70)
 


CELL 7: Gap Stability Test (Category × Year Interaction)


In [20]:
# Step 1: Prepare data
reg_df = df.dropna(subset=["written_total", "interview"]).copy()
reg_df["year_centered"] = reg_df["year"] - reg_df["year"].mean()  # center year for stability
reg_df["category_str"] = reg_df["category"].astype(str)

In [21]:
# Step 2: Fit FULL model (with interaction terms)
print("\n── Full model: interview ~ written + category + year + category:year ──")
formula_full = "interview ~ written_total + C(category_str, Treatment('General')) * year_centered"
model_full = smf.ols(formula_full, data=reg_df).fit()
print(model_full.summary().tables[1])


── Full model: interview ~ written + category + year + category:year ──
                                                                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------------------------------------------------------
Intercept                                                    233.6226      4.791     48.759      0.000     224.230     243.016
C(category_str, Treatment('General'))[T.EWS]                  -8.0051      0.773    -10.355      0.000      -9.521      -6.490
C(category_str, Treatment('General'))[T.OBC]                  -7.4694      0.567    -13.179      0.000      -8.580      -6.358
C(category_str, Treatment('General'))[T.SC]                  -11.8696      0.715    -16.611      0.000     -13.270     -10.469
C(category_str, Treatment('General'))[T.ST]                  -12.2779      0.898    -13.678      0.000     -14.038     -10.518
written_total                         

In [22]:
# Step 3: Fit RESTRICTED model (no interaction)
formula_restricted = "interview ~ written_total + C(category_str, Treatment('General')) + year_centered"
model_restricted = smf.ols(formula_restricted, data=reg_df).fit()

In [23]:
from scipy.stats import f as f_dist
 
sse_full = model_full.ssr
sse_rest = model_restricted.ssr
q = model_full.df_model - model_restricted.df_model  # number of interaction terms
n = len(reg_df)
k_full = model_full.df_model + 1
 
F_interaction = ((sse_rest - sse_full) / q) / (sse_full / model_full.df_resid)
p_interaction = 1 - f_dist.cdf(F_interaction, q, model_full.df_resid)
 
print(f"\n── Joint F-test for interaction terms ──")
print(f"F({q:.0f}, {model_full.df_resid:.0f}) = {F_interaction:.4f}")
print(f"p-value = {p_interaction:.6f}")
print(f"Result: {'SIGNIFICANT — gap is changing over time' if p_interaction < 0.05 else 'NOT SIGNIFICANT — gap is stable'}")
 


── Joint F-test for interaction terms ──
F(4, 5341) = 3.9230
p-value = 0.003492
Result: SIGNIFICANT — gap is changing over time


In [24]:
# Step 5: Extract individual interaction coefficients
print(f"\n── Individual interaction terms (Category × Year) ──")
print(f"{'Term':<35} {'β':>8} {'SE':>8} {'t':>8} {'p':>10} {'95% CI':>20} {'Interpretation'}")
print("-" * 120)
 
interaction_results = []
for param_name in model_full.params.index:
    if "year_centered" in param_name and "category_str" in param_name:
        # Extract category name from parameter
        cat = param_name.split("[T.")[1].split("]")[0] if "[T." in param_name else param_name
        beta = model_full.params[param_name]
        se = model_full.bse[param_name]
        t_val = model_full.tvalues[param_name]
        p_val = model_full.pvalues[param_name]
        ci = model_full.conf_int().loc[param_name]
 
        sig = p_val < 0.05
        direction = "narrowing" if beta > 0 else "widening"
        interp = f"Gap {direction} by {abs(beta):.2f} marks/year" if sig else "No significant change"
 
        print(f"{cat + ' × Year':<35} {beta:>8.4f} {se:>8.4f} {t_val:>8.4f} {p_val:>10.6f} [{ci[0]:.4f}, {ci[1]:.4f}]  {interp}")
 
        interaction_results.append({
            "category": cat, "beta": round(beta, 4), "se": round(se, 4),
            "t": round(t_val, 4), "p_value": round(p_val, 6),
            "ci_lo": round(ci[0], 4), "ci_hi": round(ci[1], 4),
            "significant": sig, "interpretation": interp
        })
 
interaction_df = pd.DataFrame(interaction_results)
interaction_df.to_csv(OUT_DIR / "gap_stability_interactions.csv", index=False)
 
print(f"\n✓ Gap stability analysis complete. Results saved to gap_stability_interactions.csv")
 


── Individual interaction terms (Category × Year) ──
Term                                       β       SE        t          p               95% CI Interpretation
------------------------------------------------------------------------------------------------------------------------
EWS × Year                            0.4541   0.4592   0.9889   0.322754 [-0.4461, 1.3542]  No significant change
OBC × Year                           -0.2855   0.3294  -0.8668   0.386085 [-0.9312, 0.3602]  No significant change
SC × Year                             0.7046   0.4003   1.7601   0.078443 [-0.0802, 1.4894]  No significant change
ST × Year                             1.5220   0.5187   2.9341   0.003360 [0.5051, 2.5390]  Gap narrowing by 1.52 marks/year

✓ Gap stability analysis complete. Results saved to gap_stability_interactions.csv


In [25]:
print("\n" + "=" * 70)
print("CELL 8: Confidence Intervals on Key Effect Sizes")
print("=" * 70)


CELL 8: Confidence Intervals on Key Effect Sizes


In [26]:
# Step 1: Regression β CIs (already available from statsmodels)
print("\n── Regression coefficients with 95% CIs ──")
print(f"Model: interview ~ written_total + C(category) + year_centered")
print(f"Reference category: General\n")
 
ci_df = model_restricted.conf_int()
ci_df.columns = ["ci_lo", "ci_hi"]
ci_df["beta"] = model_restricted.params
ci_df["se"] = model_restricted.bse
ci_df["t"] = model_restricted.tvalues
ci_df["p_value"] = model_restricted.pvalues
 
print(f"{'Variable':<45} {'β':>8} {'SE':>7} {'95% CI':>22} {'p':>10}")
print("-" * 100)
for idx in ci_df.index:
    row = ci_df.loc[idx]
    label = idx.replace("C(category_str, Treatment('General'))[T.", "").rstrip("]")
    print(f"{label:<45} {row['beta']:>8.4f} {row['se']:>7.4f}  [{row['ci_lo']:>8.4f}, {row['ci_hi']:>8.4f}] {row['p_value']:>10.6f}")
 
# Save regression CIs
reg_ci_results = []
for idx in ci_df.index:
    row = ci_df.loc[idx]
    label = idx.replace("C(category_str, Treatment('General'))[T.", "").rstrip("]")
    reg_ci_results.append({
        "variable": label, "beta": round(row["beta"], 4), "se": round(row["se"], 4),
        "ci_lo": round(row["ci_lo"], 4), "ci_hi": round(row["ci_hi"], 4),
        "p_value": round(row["p_value"], 8)
    })
reg_ci_df = pd.DataFrame(reg_ci_results)
 


── Regression coefficients with 95% CIs ──
Model: interview ~ written_total + C(category) + year_centered
Reference category: General

Variable                                             β      SE                 95% CI          p
----------------------------------------------------------------------------------------------------
Intercept                                     233.7456  4.7955  [224.3445, 243.1467]   0.000000
EWS                                            -8.0182  0.7739  [ -9.5354,  -6.5011]   0.000000
OBC                                            -7.5046  0.5673  [ -8.6168,  -6.3924]   0.000000
SC                                            -11.8756  0.7153  [-13.2780, -10.4733]   0.000000
ST                                            -12.3083  0.8986  [-14.0699, -10.5467]   0.000000
written_total                                  -0.0665  0.0061  [ -0.0783,  -0.0546]   0.000000
year_centered                                   2.5090  0.1326  [  2.2492,   2.7689]   0.0

In [27]:
# Step 2: Bootstrap CIs for Cohen's d
print("\n── Bootstrap 95% CIs for Cohen's d (5000 iterations) ──")
 
def bootstrap_cohens_d(group1, group2, n_bootstrap=5000, seed=42):
    """Compute Cohen's d with 95% bootstrap confidence interval."""
    rng = np.random.RandomState(seed)
    d_observed = cohens_d(group1, group2)
 
    bootstrap_ds = []
    for _ in range(n_bootstrap):
        g1_boot = rng.choice(group1, size=len(group1), replace=True)
        g2_boot = rng.choice(group2, size=len(group2), replace=True)
        d_boot = cohens_d(g1_boot, g2_boot)
        if not np.isnan(d_boot):
            bootstrap_ds.append(d_boot)
 
    ci_lo = np.percentile(bootstrap_ds, 2.5)
    ci_hi = np.percentile(bootstrap_ds, 97.5)
    return d_observed, ci_lo, ci_hi
 
gen_interview = df[df["category"] == "General"]["interview"].dropna().values
 
cohens_d_results = []
print(f"\n{'Comparison':<25} {'d':>8} {'95% CI':>22} {'Effect Size'}")
print("-" * 70)
 
for cat in ["OBC", "SC", "ST", "EWS"]:
    cat_interview = df[df["category"] == cat]["interview"].dropna().values
    d_obs, ci_lo, ci_hi = bootstrap_cohens_d(gen_interview, cat_interview)
 
    size_label = "Large" if abs(d_obs) >= 0.8 else "Medium" if abs(d_obs) >= 0.5 else "Small-medium" if abs(d_obs) >= 0.35 else "Small"
    print(f"General vs {cat:<15} {d_obs:>8.4f}  [{ci_lo:>8.4f}, {ci_hi:>8.4f}]  {size_label}")
 
    cohens_d_results.append({
        "comparison": f"General vs {cat}",
        "cohens_d": round(d_obs, 4),
        "ci_lo": round(ci_lo, 4), "ci_hi": round(ci_hi, 4),
        "effect_size": size_label
    })
 
cohens_d_ci_df = pd.DataFrame(cohens_d_results)
 
# Save both CI tables
reg_ci_df.to_csv(OUT_DIR / "regression_confidence_intervals.csv", index=False)
cohens_d_ci_df.to_csv(OUT_DIR / "cohens_d_confidence_intervals.csv", index=False)
 
print(f"\n✓ CI analysis complete. Saved to regression_confidence_intervals.csv and cohens_d_confidence_intervals.csv")
 


── Bootstrap 95% CIs for Cohen's d (5000 iterations) ──

Comparison                       d                 95% CI Effect Size
----------------------------------------------------------------------
General vs OBC               0.3459  [  0.2791,   0.4150]  Small
General vs SC                0.5191  [  0.4352,   0.6068]  Medium
General vs ST                0.5449  [  0.4365,   0.6599]  Medium
General vs EWS               0.4009  [  0.3069,   0.4957]  Small-medium

✓ CI analysis complete. Saved to regression_confidence_intervals.csv and cohens_d_confidence_intervals.csv


In [28]:
print("\n" + "=" * 70)
print("CELL 9: Practical Significance — Rank Impact Simulation")
print("=" * 70)


CELL 9: Practical Significance — Rank Impact Simulation


In [29]:
# Step 1: Measure how tight rankings are (consecutive rank gaps)
print("\n── How tight are UPSC rankings? ──")
 
margin_results = []
for year in sorted(df["year"].unique()):
    ydf = df[df["year"] == year].sort_values("final_total", ascending=False).reset_index(drop=True)
    diffs = ydf["final_total"].diff(-1).dropna().abs()
 
    margin_results.append({
        "year": int(year),
        "n_candidates": len(ydf),
        "median_gap_between_ranks": round(diffs.median(), 1),
        "mean_gap_between_ranks": round(diffs.mean(), 1),
        "pct_within_5_marks": round((diffs <= 5).sum() / len(diffs) * 100, 1),
        "pct_within_10_marks": round((diffs <= 10).sum() / len(diffs) * 100, 1),
    })
 
margin_df = pd.DataFrame(margin_results)
print(margin_df.to_string(index=False))
print(f"\n→ ~{margin_df['pct_within_5_marks'].mean():.0f}% of consecutive ranks are within 5 marks of each other!")
 


── How tight are UPSC rankings? ──
 year  n_candidates  median_gap_between_ranks  mean_gap_between_ranks  pct_within_5_marks  pct_within_10_marks
 2020           761                       0.0                     0.8                98.4                 98.6
 2021           685                       0.0                     0.6                98.7                 99.1
 2022           933                       0.0                     0.5                98.3                 98.9
 2023          1016                       0.0                     0.5                99.1                 99.5
 2024           999                       0.0                     0.3                99.1                 99.5
 2025           958                       0.0                     0.5                98.9                 99.2

→ ~99% of consecutive ranks are within 5 marks of each other!


In [30]:
# Step 2: Simulate closing the gap, year by year
print("\n── Rank impact simulation: What if the gap didn't exist? ──")
 
simulation_results = []
 
for year in sorted(df["year"].unique()):
    ydf = df[df["year"] == year].copy()
 
    # Current rankings
    ydf = ydf.sort_values("final_total", ascending=False).reset_index(drop=True)
    ydf["rank_actual"] = range(1, len(ydf) + 1)
 
    # General mean for this year
    gen_mean = ydf[ydf["category"] == "General"]["interview"].mean()
 
    for cat in ["OBC", "SC", "ST", "EWS"]:
        cat_mask = ydf["category"] == cat
        n_cat = cat_mask.sum()
        if n_cat == 0:
            continue
 
        # Observed gap
        cat_mean = ydf[cat_mask]["interview"].mean()
        gap = gen_mean - cat_mean
 
        # Simulate: add gap to this category's interview & final marks
        ydf_sim = ydf.copy()
        ydf_sim.loc[cat_mask, "interview"] = ydf_sim.loc[cat_mask, "interview"] + gap
        ydf_sim.loc[cat_mask, "final_total"] = ydf_sim.loc[cat_mask, "final_total"] + gap
 
        # Recalculate ranks
        ydf_sim = ydf_sim.sort_values("final_total", ascending=False).reset_index(drop=True)
        ydf_sim["rank_sim"] = range(1, len(ydf_sim) + 1)
 
        # Merge to compare
        merged = ydf[["roll_no", "rank_actual"]].merge(
            ydf_sim[["roll_no", "rank_sim"]], on="roll_no"
        )
        cat_merged = merged[merged["roll_no"].isin(ydf[cat_mask]["roll_no"])]
        rank_improvement = cat_merged["rank_actual"] - cat_merged["rank_sim"]
 
        # Top-100 and Top-50 impact
        top100_actual = (cat_merged["rank_actual"] <= 100).sum()
        top100_sim = (cat_merged["rank_sim"] <= 100).sum()
        top50_actual = (cat_merged["rank_actual"] <= 50).sum()
        top50_sim = (cat_merged["rank_sim"] <= 50).sum()
 
        simulation_results.append({
            "year": int(year),
            "category": cat,
            "n": n_cat,
            "gap_marks": round(gap, 2),
            "avg_rank_improvement": round(rank_improvement.mean(), 1),
            "median_rank_improvement": round(rank_improvement.median(), 1),
            "max_rank_improvement": int(rank_improvement.max()),
            "candidates_who_improved": int((rank_improvement > 0).sum()),
            "top100_actual": top100_actual,
            "top100_simulated": top100_sim,
            "top100_gained": top100_sim - top100_actual,
            "top50_actual": top50_actual,
            "top50_simulated": top50_sim,
            "top50_gained": top50_sim - top50_actual,
        })
 
simulation_df = pd.DataFrame(simulation_results)
 


── Rank impact simulation: What if the gap didn't exist? ──


In [31]:
# Step 3: Print results
print(f"\n{'Year':<6} {'Category':<6} {'Gap':>6} {'N':>5} {'Avg Rank↑':>10} {'Max Rank↑':>10} {'Top100 Gained':>14}")
print("-" * 65)
for _, row in simulation_df.iterrows():
    print(f"{row['year']:<6} {row['category']:<6} {row['gap_marks']:>+6.1f} {row['n']:>5} "
          f"{row['avg_rank_improvement']:>10.1f} {row['max_rank_improvement']:>10} "
          f"{row['top100_gained']:>14}")
 


Year   Category    Gap     N  Avg Rank↑  Max Rank↑  Top100 Gained
-----------------------------------------------------------------
2020   OBC      +5.2   229       18.6         49              4
2020   SC      +12.1   122       48.0        105              1
2020   ST      +12.4    61       66.2        107              0
2020   EWS      +9.1    86       53.0         85              0
2021   OBC      +5.3   203       18.5         44              2
2021   SC       +7.8   105       28.4         71              2
2021   ST      +13.4    60       55.7        107              1
2021   EWS      +7.8    73       37.8         70              4
2022   OBC      +4.3   263       23.8         53              3
2022   SC      +10.1   154       52.6        131              2
2022   ST       +9.4    72       63.3        118              1
2022   EWS      +3.3    99       26.8         46              1
2023   OBC      +7.6   303       45.7         97              4
2023   SC       +7.1   165       45

In [32]:
# Step 4: Aggregate across years
print("\n── 6-Year Aggregate Impact ──")
agg = simulation_df.groupby("category").agg({
    "gap_marks": "mean",
    "avg_rank_improvement": "mean",
    "max_rank_improvement": "max",
    "top100_gained": "sum",
    "top50_gained": "sum",
}).round(1)
agg.columns = ["Avg Gap (marks)", "Avg Rank Improvement", "Max Single-Year Improvement",
                "Total Top-100 Gained (6yr)", "Total Top-50 Gained (6yr)"]
print(agg)
 
# Save results
margin_df.to_csv(OUT_DIR / "rank_margin_analysis.csv", index=False)
simulation_df.to_csv(OUT_DIR / "rank_impact_simulation.csv", index=False)
 
print(f"\n✓ Practical significance analysis complete.")
print(f"  Saved: rank_margin_analysis.csv, rank_impact_simulation.csv")
print(f"\n  Key takeaway: Closing the gap would shift SC candidates by ~{agg.loc['SC', 'Avg Rank Improvement']:.0f} "
      f"rank positions and ST by ~{agg.loc['ST', 'Avg Rank Improvement']:.0f} positions on average.")
 


── 6-Year Aggregate Impact ──
          Avg Gap (marks)  Avg Rank Improvement  Max Single-Year Improvement  \
category                                                                       
EWS                   6.9                  48.5                          120   
OBC                   5.9                  30.0                           97   
SC                    9.1                  45.8                          147   
ST                    9.6                  55.2                          152   

          Total Top-100 Gained (6yr)  Total Top-50 Gained (6yr)  
category                                                         
EWS                               14                          5  
OBC                               21                         15  
SC                                11                          5  
ST                                 3                          3  

✓ Practical significance analysis complete.
  Saved: rank_margin_analysis.csv, rank_impact_